# Ablation Study



In [ ]:
import os
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    !git clone https://github.com/zamsi-ajegetina/landmark-recognition.git
    %cd landmark-recognition
    !pip install -q -r requirements.txt
    from google.colab import drive
    drive.mount('/content/drive')

    ! pip install gdown -q
    ! gdown 10hLpehomjhFJTNbJkCRyc4lkV4Iaouan --output landmark_images.zip

    !unzip landmark_images.zip -d landmark_images
    !rm landmark_images.zip

In [ ]:
import wandb
wandb.login()

In [ ]:
%matplotlib inline
import sys, torch
sys.path.insert(0, '..')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
!python ablation_runner.py --config configs/A0_scratch.yaml

In [ ]:
!python ablation_runner.py --config configs/A1_resnet18_frozen.yaml

In [ ]:
!python ablation_runner.py --config configs/A2_resnet18_full.yaml

In [ ]:
!python ablation_runner.py --config configs/A4_resnet50_full.yaml

In [ ]:
!python ablation_runner.py --config configs/A6_vitb16_frozen.yaml

In [ ]:
!python ablation_runner.py --config configs/A7_resnet50_minimal_aug.yaml

In [ ]:
!python ablation_runner.py --config configs/A8_resnet50_weighted.yaml

## 5. Results table

In [ ]:
import torch
from pathlib import Path
from src.data import get_data_loaders
from src.transfer import get_model_transfer_learning
from src.model import CustomModel
from src.optimization import get_loss
from src.train import one_epoch_test
from sklearn.metrics import f1_score

data_loaders = get_data_loaders(batch_size=64, num_workers=2)
loss_fn = get_loss()

def eval_run(ckpt, model_fn):
    if not Path(ckpt).exists():
        return None, None, None
    m = model_fn()
    m.load_state_dict(torch.load(ckpt, map_location='cpu'))
    _, top1, top5, targets, preds = one_epoch_test(data_loaders['test'], m, loss_fn)
    mf1 = f1_score(targets, preds, average='macro', zero_division=0)
    return top1, top5, mf1

runs = [
    ('A0 scratch',       'checkpoints/A0_scratch.pt',          lambda: CustomModel(50)),
    ('A1 RN18 frozen',   'checkpoints/A1_resnet18_frozen.pt',  lambda: get_model_transfer_learning('resnet18', 50, 'frozen')),
    ('A2 RN18 full',     'checkpoints/A2_resnet18_full.pt',    lambda: get_model_transfer_learning('resnet18', 50, 'full')),
    ('A4 RN50 full',     'checkpoints/A4_resnet50_full.pt',    lambda: get_model_transfer_learning('resnet50', 50, 'full')),
    ('A6 ViT-B/16',      'checkpoints/A6_vitb16_frozen.pt',    lambda: get_model_transfer_learning('vit_b_16', 50, 'frozen')),
    ('A7 RN50 min aug',  'checkpoints/A7_resnet50_minimal_aug.pt', lambda: get_model_transfer_learning('resnet50', 50, 'full')),
    ('A8 RN50 WRS',      'checkpoints/A8_resnet50_weighted.pt',    lambda: get_model_transfer_learning('resnet50', 50, 'full')),
]

print(f'{"Run":<22} {"Top-1":>7} {"Top-5":>7} {"Macro-F1":>10}')
print('─' * 50)
for name, ckpt, model_fn in runs:
    top1, top5, mf1 = eval_run(ckpt, model_fn)
    if top1 is None:
        print(f'{name:<22} (checkpoint not found)')
    else:
        print(f'{name:<22} {100*top1:6.1f}% {100*top5:6.1f}% {100*mf1:9.1f}%')

## 6. Ablation analysis plots

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

results = {
    'A0\nScratch': (34.2, 65, 30.0),
    'A1\nRN18 frozen': (73.1, 92.1, 72.9),
    'A2\nRN18 full': (81.1,93.7,81.2),
    'A4\nRN50 full': (85.4, 94.7, 85.3),  
    'A6\nViT-B/16': (82.4, 94.6, 82.2),
    'A7\nMin aug': (84.3, 95.1, 84.3),
    'A8\nWRS': (69.8, 76.8, 63.2),
}

labels = [k for k, v in results.items() if v[0] is not None]
top1   = [results[k][0] for k in labels]
top5   = [results[k][1] for k in labels]
mf1    = [results[k][2] for k in labels]

x = np.arange(len(labels))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - width, top1, width, label='Top-1 %', color='steelblue')
ax.bar(x,         top5, width, label='Top-5 %', color='seagreen')
ax.bar(x + width, mf1,  width, label='Macro-F1 %', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylim(0, 100)
ax.set_ylabel('Accuracy / F1 (%)')
ax.set_title('Ablation results — African Landmark Recognition')
ax.legend()
ax.axhline(y=60, color='gray', linestyle='--', linewidth=0.8, label='Passing threshold')
plt.tight_layout()
plt.savefig('ablation_comparison.png', dpi=150)
plt.show()
print('Figure saved as ablation_comparison.png')